In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from torch.nn import functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from domain_shift.core.config import settings
from domain_shift.CycleGAN.triplet_data_loader_val import get_data_loader
from domain_shift.CycleGAN.triplet_models_val import CycleGAN
from domain_shift.data_extraction.process_DRIAMS import DRIAMS_bin_to_df

In [ ]:
# Load the data
driams_1 = DRIAMS_bin_to_df(settings.DRIAMS_C_PATH)
driams_2 = DRIAMS_bin_to_df(settings.DRIAMS_D_PATH)
driams_3 = DRIAMS_bin_to_df(settings.DRIAMS_B_PATH)

In [5]:
# Combine the 'species' columns from both datasets
combined_species = pd.concat([driams_1['species'], driams_2['species']])

# Find the most represented species across both datasets
species_counts = combined_species.value_counts()
top_species = species_counts.index[:2]
top_species = top_species.tolist()
top_species

['Escherichia coli', 'Staphylococcus aureus']

In [6]:
# Show the distribution of the most represented species in each dataset
driams_1['species'].value_counts()[top_species], \
driams_2['species'].value_counts()[top_species]

(species
 Escherichia coli         927
 Staphylococcus aureus    738
 Name: count, dtype: int64,
 species
 Escherichia coli         2013
 Staphylococcus aureus    2174
 Name: count, dtype: int64)

In [7]:
# Filter by the 4 species with most representation
filtered_driams_1 = driams_1[driams_1["species"].isin(top_species)]
filtered_driams_2 = driams_2[driams_2["species"].isin(top_species)]
filtered_driams_3 = driams_3[driams_3["species"].isin(top_species)]

In [8]:
# Train Random Forest on DRIAMS 1
X = np.vstack(filtered_driams_1["binned_6000"].values)
y = filtered_driams_1["species"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

f1_score(y_test, y_pred, average='weighted')

1.0

In [9]:
# Test the model on the filtered driams 2
X = np.vstack(filtered_driams_2["binned_6000"].values)
y = filtered_driams_2["species"].values

y_pred = rf.predict(X)

f1_score(y, y_pred, average='weighted')

0.8653695773662051

In [10]:
# Test the model on the filtered driams 3
X = np.vstack(filtered_driams_3["binned_6000"].values)
y = filtered_driams_3["species"].values

y_pred = rf.predict(X)

f1_score(y, y_pred, average='weighted')

1.0